# Motor de Simulación Salarial
### Preprocesamiento de Datos — Actividades 1 al 5
**Estudiantes:** Breyder David Cabrera · Jessica Malpud  
**Dataset:** Global AI Jobs & Skills Demand Dataset (2020–2026)  
**Fuente:** https://www.kaggle.com/datasets/chaitanyajamble/global-ai-jobs-and-skills-demand-dataset-20202026/data

## Importaciones

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Carga del dataset

In [2]:
df = pd.read_csv('global_ai_jobs_dataset.csv')
print(f'Filas: {df.shape[0]} | Columnas: {df.shape[1]}')
df.head()

Filas: 120000 | Columnas: 16


,job_id,job_title,industry,company_size,country,city,experience_level,remote_type,education_required,ai_domain,salary_usd,years_experience_required,benefits_score,posting_date,skills,tools_used
0,1,AI Researcher,Manufacturing,SME,Canada,Berlin,Senior,Remote,PhD,Generative AI,104015,4,1.98,2021-04-01,Python;Computer Vision;SQL;NLP,Kubernetes;PyTorch;AWS
1,2,MLOps Engineer,Education,Startup,India,Tokyo,Entry,Remote,Bachelor,Recommendation Systems,103263,6,3.67,2020-04-12,NLP;PyTorch;Data Analysis;Computer Vision,Python;Spark;Docker
2,3,Data Analyst,E-commerce,SME,United Kingdom,Bangalore,Mid,Hybrid,Bachelor,Robotics,104190,7,2.26,2023-01-31,NLP;Computer Vision;Python;PyTorch,Python;Kubernetes;PyTorch
3,4,NLP Engineer,Technology,Enterprise,Brazil,Amsterdam,Mid,Remote,PhD,Recommendation Systems,51440,9,2.26,2022-09-30,Data Analysis;Statistics;TensorFlow;Python,Python;PyTorch;TensorFlow
4,5,AI Researcher,E-commerce,SME,Netherlands,Amsterdam,Senior,Hybrid,Master,Computer Vision,58358,0,2.44,2022-07-03,NLP;Machine Learning;PyTorch;SQL,TensorFlow;SQL;Azure


In [3]:
df.describe()

,job_id,salary_usd,years_experience_required,benefits_score
count,120000.000000,120000.000000,120000.000000,120000.000000
mean,60000.500000,119995.002358,4.487683,2.995962
std,34641.160489,46150.388458,2.874546,1.153465
min,1.000000,40001.000000,0.000000,1.000000
25%,30000.750000,80165.750000,2.000000,2.000000
50%,60000.500000,119914.500000,4.000000,2.990000
75%,90000.250000,159840.000000,7.000000,3.990000
max,120000.000000,199999.000000,9.000000,5.000000


In [4]:
# Identificar variables categóricas
categorical_cols = df.select_dtypes(include=['object']).columns
print("Variables categóricas:", list(categorical_cols))

# Calcular la moda para cada variable categórica
modes = {}
for col in categorical_cols:
    mode_values = df[col].mode()
    modes[col] = mode_values.tolist() if not mode_values.empty else None

# Mostrar las modas
for col, mode in modes.items():
    print(f"Moda de {col}: {mode}")

/tmp/ipykernel_8548/4266739755.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns


Variables categóricas: ['job_title', 'industry', 'company_size', 'country', 'city', 'experience_level', 'remote_type', 'education_required', 'ai_domain', 'posting_date', 'skills', 'tools_used']
Moda de job_title: ['Prompt Engineer']
Moda de industry: ['Finance']
Moda de company_size: ['Enterprise']
Moda de country: ['Germany']
Moda de city: ['London']
Moda de experience_level: ['Mid']
Moda de remote_type: ['Onsite']
Moda de education_required: ['Bachelor']
Moda de ai_domain: ['Computer Vision']
Moda de posting_date: ['2025-11-02']
Moda de skills: ['SQL;Machine Learning;PyTorch;Deep Learning', 'SQL;TensorFlow;Machine Learning;Deep Learning']
Moda de tools_used: ['Python;Docker;Azure']


# Problemas del Conjunto de datos

El conjunto de datos presenta varias limitaciones que deben ser consideradas antes de desarrollar el modelo de regresión y el posterior motor de simulación. En primer lugar, se observa una alta predominancia de variables categóricas, como el cargo, la industria, el país o la ciudad, lo que implica la necesidad de aplicar técnicas de codificación (como variables dummy). Este proceso puede incrementar significativamente la dimensionalidad del modelo, generando riesgos de sobreajuste y reduciendo su capacidad de generalización.

Adicionalmente, la variable de habilidades (skills) se encuentra almacenada como texto no estructurado, lo que dificulta su uso directo en el modelo. Para incorporarla, es necesario transformarla en múltiples variables binarias, lo cual no solo incrementa la complejidad del análisis, sino que también puede introducir problemas de multicolinealidad, dado que muchas habilidades tienden a estar altamente correlacionadas entre sí.

Por otra parte, se identifican posibles inconsistencias internas en los datos, como combinaciones poco coherentes entre nivel de experiencia, años requeridos y salario. Este tipo de ruido puede afectar la estimación de los coeficientes del modelo y, en consecuencia, distorsionar los resultados del ejercicio de simulación.

Asimismo, el dataset presenta un problema de variables omitidas, ya que no incluye factores relevantes como habilidades blandas, calidad de la formación, redes de contacto o características específicas de las empresas. La ausencia de estas variables puede generar sesgos en las estimaciones, limitando la capacidad explicativa del modelo.

Otro aspecto importante es la posible naturaleza artificial o simplificada de los datos, evidenciada en la distribución relativamente uniforme de los salarios y la ausencia de valores extremos. Esto sugiere que el conjunto de datos podría no reflejar completamente la complejidad del mercado laboral real, lo que afectaría la validez externa de los resultados.

Finalmente, aunque se dispone de una variable temporal (fecha de publicación), esta no se encuentra estructurada para su análisis directo, lo que implica una subutilización de la dimensión temporal. No incorporar esta información limita la posibilidad de capturar tendencias en la evolución de los salarios a lo largo del tiempo.





## Actividad 1 — Verificación y corrección de tipos de datos
Se identifican los tipos de cada variable y se corrigen los incorrectos:
- `job_id`: se elimina (identificador sin valor predictivo)
- `posting_date`: se convierte de `str` a `datetime64`
- El resto de variables tienen tipos correctos

In [5]:
print('=== Tipos originales ===')
print(df.dtypes)
print()

# job_id no aporta información predictiva
df = df.drop(columns=['job_id'])

# posting_date: string → datetime
df['posting_date'] = pd.to_datetime(df['posting_date'])

print('=== Tipos corregidos ===')
print(df.dtypes)
print()
print('=== Valores nulos ===')
print(df.isnull().sum())

=== Tipos originales ===
job_id                         int64
job_title                        str
industry                         str
company_size                     str
country                          str
city                             str
experience_level                 str
remote_type                      str
education_required               str
ai_domain                        str
salary_usd                     int64
years_experience_required      int64
benefits_score               float64
posting_date                     str
skills                           str
tools_used                       str
dtype: object

=== Tipos corregidos ===
job_title                               str
industry                                str
company_size                            str
country                                 str
city                                    str
experience_level                        str
remote_type                             str
education_required                 

## Actividad 2 — Partición Train / Test
Se divide el dataset en 80% entrenamiento y 20% prueba usando `train_test_split` de scikit-learn.  
Se usa `stratify=experience_level` para mantener la proporción de niveles de experiencia en ambos conjuntos.

In [6]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df['experience_level']
)

print(f'Train: {len(train_df):,} filas ({len(train_df)/len(df):.0%})')
print(f'Test:  {len(test_df):,} filas ({len(test_df)/len(df):.0%})')
print()
print('Distribución de experience_level en train:')
print(train_df['experience_level'].value_counts(normalize=True).round(3))
print()
print('Distribución de experience_level en test:')
print(test_df['experience_level'].value_counts(normalize=True).round(3))

Train: 96,000 filas (80%)
Test:  24,000 filas (20%)

Distribución de experience_level en train:
experience_level
Mid       0.251
Lead      0.250
Entry     0.250
Senior    0.249
Name: proportion, dtype: float64

Distribución de experience_level en test:
experience_level
Mid       0.251
Lead      0.250
Entry     0.250
Senior    0.249
Name: proportion, dtype: float64


## Actividad 3 — Guardar conjuntos en archivos separados
Se exportan los conjuntos **sin transformar** a `train.csv` y `test.csv`.

In [7]:
train_df.to_csv('train.csv', index=False)
test_df.to_csv('test.csv',  index=False)

print('Archivos guardados:')
print('  train.csv')
print('  test.csv')

Archivos guardados:
  train.csv
  test.csv


## Actividad 4 — One-Hot Encoding para variables categóricas
Se aplica `pd.get_dummies()` a las 9 variables categóricas nominales.  
De `posting_date` se extraen `posting_year` y `posting_month` como variables numéricas.  
Las columnas `skills` y `tools_used` son listas multi-valor separadas por `;` y requieren tratamiento especial (multi-label encoding), por lo que se excluyen de este paso.  
Se usa `.align()` para garantizar que train y test tengan exactamente las mismas columnas.

In [8]:
# Extraer año y mes de posting_date
for split in [train_df, test_df]:
    split['posting_year']  = split['posting_date'].dt.year
    split['posting_month'] = split['posting_date'].dt.month

train_df = train_df.drop(columns=['posting_date'])
test_df  = test_df.drop(columns=['posting_date'])

# Variables categóricas a codificar
categorical_cols = [
    'job_title', 'industry', 'company_size', 'country', 'city',
    'experience_level', 'remote_type', 'education_required', 'ai_domain'
]

# Variables texto multi-valor (se excluyen del OHE estándar)
text_cols = ['skills', 'tools_used']

train_enc = pd.get_dummies(train_df.drop(columns=text_cols),
                            columns=categorical_cols, drop_first=False)
test_enc  = pd.get_dummies(test_df.drop(columns=text_cols),
                            columns=categorical_cols, drop_first=False)

# Alinear columnas (test puede tener menos categorías que train)
train_enc, test_enc = train_enc.align(test_enc, join='left', fill_value=0)

print(f'Columnas antes del OHE: 15')
print(f'Columnas después del OHE — train: {train_enc.shape[1]} | test: {test_enc.shape[1]}')
train_enc.head()

Columnas antes del OHE: 15
Columnas después del OHE — train: 63 | test: 63


,salary_usd,years_experience_required,benefits_score,posting_year,posting_month,job_title_AI Engineer,job_title_AI Product Manager,job_title_AI Researcher,job_title_Computer Vision Engineer,job_title_Data Analyst,...,remote_type_Onsite,remote_type_Remote,education_required_Bachelor,education_required_Master,education_required_PhD,ai_domain_Computer Vision,ai_domain_Generative AI,ai_domain_NLP,ai_domain_Recommendation Systems,ai_domain_Robotics
33649,127637,1,3.45,2024,5,False,False,False,False,False,...,True,False,False,False,True,False,False,True,False,False
2732,139148,4,3.84,2021,7,True,False,False,False,False,...,False,False,False,True,False,False,False,True,False,False
94980,116727,3,1.91,2021,12,False,False,False,False,False,...,True,False,False,True,False,False,False,True,False,False
83410,102413,5,1.51,2025,10,False,False,False,False,False,...,False,False,True,False,False,False,False,False,True,False
66936,81849,5,3.35,2020,10,False,False,False,False,False,...,True,False,False,False,True,False,True,False,False,False


## Actividad 4b — Multi-Label Encoding para `skills` y `tools_used`

Las columnas `skills` y `tools_used` contienen múltiples valores por fila separados por `;`,
por ejemplo `"Python;TensorFlow;Docker"`. Esto no se puede manejar con `get_dummies()` directamente,
ya que cada fila tiene una cantidad variable de valores.

Se usa `MultiLabelBinarizer` de scikit-learn, que crea una columna binaria por cada skill/tool única:
- `1` si el perfil tiene esa skill o herramienta
- `0` si no la tiene

Al igual que con el OHE, el binarizer se ajusta (`fit`) solo con train y se aplica (`transform`)
a test para evitar *data leakage*. Los prefijos `skill_` y `tool_` se agregan para distinguir
estas columnas de las demás.

In [9]:
from sklearn.preprocessing import MultiLabelBinarizer

# ── skills ────────────────────────────────────────────────────────────────────
mlb_skills = MultiLabelBinarizer()

train_skills = train_df['skills'].str.split(';')
test_skills  = test_df['skills'].str.split(';')

train_skills_enc = pd.DataFrame(
    mlb_skills.fit_transform(train_skills),
    columns=['skill_' + c for c in mlb_skills.classes_],
    index=train_df.index
)
test_skills_enc = pd.DataFrame(
    mlb_skills.transform(test_skills),
    columns=['skill_' + c for c in mlb_skills.classes_],
    index=test_df.index
)

# ── tools_used ────────────────────────────────────────────────────────────────
mlb_tools = MultiLabelBinarizer()

train_tools = train_df['tools_used'].str.split(';')
test_tools  = test_df['tools_used'].str.split(';')

train_tools_enc = pd.DataFrame(
    mlb_tools.fit_transform(train_tools),
    columns=['tool_' + c for c in mlb_tools.classes_],
    index=train_df.index
)
test_tools_enc = pd.DataFrame(
    mlb_tools.transform(test_tools),
    columns=['tool_' + c for c in mlb_tools.classes_],
    index=test_df.index
)

# ── Unir con el dataset codificado ────────────────────────────────────────────
train_enc = pd.concat([train_enc, train_skills_enc, train_tools_enc], axis=1)
test_enc  = pd.concat([test_enc,  test_skills_enc,  test_tools_enc],  axis=1)

print(f'Columnas finales — train: {train_enc.shape[1]} | test: {test_enc.shape[1]}')

Columnas finales — train: 83 | test: 83


## Actividad 5 — Escalamiento de variables numéricas
Se aplica `StandardScaler` (media = 0, desviación estándar = 1) a las variables numéricas continuas.  
**Importante:** el scaler se ajusta (`fit`) únicamente con el conjunto de entrenamiento y luego se aplica (`transform`) al conjunto de prueba para evitar *data leakage*.  
La variable objetivo `salary_usd` **no se escala** en este paso.

In [10]:
numeric_cols = ['years_experience_required', 'benefits_score',
                'posting_year', 'posting_month']

scaler = StandardScaler()

# fit solo en train, transform en ambos
train_enc[numeric_cols] = scaler.fit_transform(train_enc[numeric_cols])
test_enc[numeric_cols]  = scaler.transform(test_enc[numeric_cols])

print('=== Estadísticos tras escalamiento (train) ===')
train_enc[numeric_cols].describe().round(4)

=== Estadísticos tras escalamiento (train) ===


,years_experience_required,benefits_score,posting_year,posting_month
count,96000.0000,96000.0000,96000.0000,96000.0000
mean,0.0000,-0.0000,-0.0000,-0.0000
std,1.0000,1.0000,1.0000,1.0000
min,-1.5610,-1.7330,-1.4643,-1.5982
25%,-0.8648,-0.8659,-0.8789,-0.7290
50%,-0.1687,0.0012,-0.2936,0.1401
75%,0.8755,0.8596,0.8771,1.0093
max,1.5716,1.7354,2.0479,1.5887


## Guardar datasets procesados
Se exportan los conjuntos transformados para usar en el entrenamiento del modelo.

In [11]:
train_enc.to_csv('train_processed.csv', index=False)
test_enc.to_csv('test_processed.csv',  index=False)

print('Archivos guardados:')
print('  train_processed.csv')
print('  test_processed.csv')
print()
print(f'Shape train procesado: {train_enc.shape}')
print(f'Shape test  procesado: {test_enc.shape}')

Archivos guardados:
  train_processed.csv
  test_processed.csv

Shape train procesado: (96000, 83)
Shape test  procesado: (120000, 83)
